# VTI Measurement — <DEVICE>

Measurement notebook template for the **VTI** rig (dgg-vti): FastBi1K5 magnet
(Ethernet), Lakeshore 335, SR830 lock-ins (sr0-sr3), two Keithley 2400 gates,
SRS DC205 top gate. Sweep stack: spxtr/measureme `sweep` (`sweep.AsyncStation`).

`metadata.hera.policy = append_only` — this notebook is the scientific record:
append new cells; don't edit or delete executed ones.

Cells marked **PATTERN** are runnable-shaped examples with `<PLACEHOLDER>`
values — fill them in deliberately before running; never run them blind.

## Setup

In [ ]:
import os
import time

import numpy as np
import scipy.constants as cst

import qcodes
from qcodes import validators
from qcodes.instrument_drivers.stanford_research import SR830
from qcodes.instrument_drivers.Keithley import Keithley2400

import sweep

from sharpelab_nb.constants import VTI
from sharpelab_nb.thermometry import temperature_from_logfiles

from drivers.FastBi1K5 import FastBi1K5
from drivers.Lakeshore_335 import Lakeshore_335
from drivers.SRS_DC205 import SRS_DC205
from drivers.qdac import QDac2, bring_up_qdac_gate

In [ ]:
main_dir = os.getcwd()
sweep.set_basedir(os.path.join(main_dir, "data"))
main_dir

# Connect Instruments

## Magnet

In [ ]:
magnet = FastBi1K5('magnet', '192.168.0.10')  # <-- verify address
magnet.status()

## Lakeshore

In [ ]:
# heater: LS335 control only (setpoints / ranges / heater power). The lakeshore
# worker (VTI Monitor launcher) is the sole GPIB master on the LS335; per-point
# temperature is read from its logfiles, never over GPIB. fp() these params.
heater = Lakeshore_335('l335_3', 'GPIB0::10::INSTR')

T_sample = temperature_from_logfiles("T_sample", VTI.temp_logfile_candidates["T_sample"])
T_vaporizer = temperature_from_logfiles("T_vaporizer", VTI.temp_logfile_candidates["T_vaporizer"])

print(f"Sample:    {T_sample()} K")
print(f"Vaporizer: {T_vaporizer()} K")

## Lock-ins

In [ ]:
sr0 = SR830('sr0', "GPIB0::17::INSTR")  # <-- verify — current channel (Ithaco preamp output)
sr1 = SR830('sr1', "GPIB0::14::INSTR")  # <-- verify — also the AC drive source
sr2 = SR830('sr2', "GPIB0::19::INSTR")  # <-- verify
sr3 = SR830('sr3', "GPIB0::18::INSTR")  # <-- verify

# Warm the qcodes cache for time_constant + filter_slope so they land in
# the run-metadata instruments snapshot (the snapshot serializes cached
# values; without a read they come back None and tau is unrecoverable from
# disk). Cheap GPIB query on the SR830. The swept-point dwell floor is
# k*tau (STYLE rule 12), so record tau at bring-up.
for _sr in (sr0, sr1, sr2, sr3):
    print(f"  {_sr.name}: tau={_sr.time_constant()} s, slope={_sr.filter_slope()} dB/oct")

## Keithleys

In [ ]:
k_bg = Keithley2400('k_bg', 'GPIB0::24::INSTR')  # <-- verify address — back gate
k_bg.mode.set('VOLT')
k_bg.volt.set(0)
k_bg.rangei.set(1e-6)
k_bg.rangev.set(21)
k_bg.volt.step = 0.005

In [ ]:
k_2410 = Keithley2400('k_2410', 'GPIB0::25::INSTR')  # <-- verify address — Si global gate
k_2410.mode.set('VOLT')
k_2410.volt.set(0)
k_2410.rangei.set(1e-6)
k_2410.rangev.set(21)
k_2410.volt.step = 0.005

## DC205 (top gate)

In [ ]:
k_tg = SRS_DC205("k_tg", "ASRL7::INSTR")  # <-- verify address — top gate
k_tg.ramp_rate = 2  # V/s

In [ ]:
# Sweepable top-gate parameter — appears in the data as column 'DC_bias'.
def dcbias(vg):
    k_tg.ramp_voltage(vg)

dc205_volt = qcodes.Parameter('DC_bias', unit='V', set_cmd=dcbias)

## QDAC (optional gate source)

QDevil QDAC-II — low-voltage (**±10 V**) gate source for the hBN gates (tg/bg).
**Off by default**: set `QDAC["enabled"] = True`, fill in the address, and list
the gate channels when the QDAC is wired in (it feeds the QSwitch source-side
Fischer). The Si global gate stays on the high-voltage Keithley 2410 — the
QDAC's ±10 V cannot reach it, and `bring_up_qdac_gate` rejects a fence that
exceeds the output range.

Each QDAC channel reads its own current, so a gate drives the generic gate-
leakage interrupter directly (no Keithley-specific guard):

    from sharpelab_nb import LeakageDetector, leakage_check
    det = LeakageDetector()
    s.register_run_before(leakage_check, args=(s, det, qd_bg.read_current_A, qd_bg.dc_constant_V))

In [ ]:
# QDAC-II gate source (optional). Fill in address + gates, then flip enabled.
# Each gate maps a name -> bring_up_qdac_gate kwargs: channel + limit required;
# output_range ('high' ±10 V / 'low' ±2 V), step, slew, output_filter,
# meas_range, meas_nplc are optional. Limits must fit the output range.
QDAC = {
    "enabled": False,
    "address": "<verify address>",  # e.g. "TCPIP::192.168.0.XX::5025::SOCKET"
    "gates": {
        # "qd_tg": {"channel": 1, "limit": (-6.5, 7.5)},
        # "qd_bg": {"channel": 2, "limit": (-5.0, 4.8), "meas_range": "low", "meas_nplc": 10},
    },
}

qdac = None
if QDAC["enabled"]:
    qdac = QDac2("qdac", QDAC["address"])
    for _name, _cfg in QDAC["gates"].items():
        _cfg = dict(_cfg)
        _ch = bring_up_qdac_gate(qdac.channel(_cfg.pop("channel")), **_cfg)
        globals()[_name] = _ch
        print(f"  {_name}: {_ch.name}  fence={_cfg['limit']}  V={_ch.dc_constant_V():+.4f}")

# SAFETIES

Run immediately after instrument bring-up, before any sweep.

In [ ]:
# Gate validators — hard limits enforced at the qcodes parameter level.
# Set per device/cooldown BEFORE any sweeping.
TG_MIN, TG_MAX = <TG_MIN>, <TG_MAX>  # top gate (dc205_volt / k_tg), V
BG_MIN, BG_MAX = <BG_MIN>, <BG_MAX>  # back gate (k_bg), V
SI_MIN, SI_MAX = <SI_MIN>, <SI_MAX>  # Si global gate (k_2410), V

dc205_volt.vals = validators.Numbers(TG_MIN, TG_MAX)
k_bg.volt.vals = validators.Numbers(BG_MIN, BG_MAX)
k_2410.volt.vals = validators.Numbers(SI_MIN, SI_MAX)

# Limit lists for sanitize_tg_bg()
tg_lims, bg_lims = [TG_MIN, TG_MAX], [BG_MIN, BG_MAX]

# TODO: magnet limit pending FastBi1K5 API review

# Helpers

In [ ]:
def spoints(a, b, c):
    return np.linspace(a, b, c)


def sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims):
    """
    Given a list of tg and bg voltages, removes any points which are outside tg_lims and bg_lims
    """
    vtg_new_vec = []
    vbg_new_vec = []
    for ind in range(len(vtg_vec)):
        vtg, vbg = vtg_vec[ind], vbg_vec[ind]
        if vtg < tg_lims[0] or vtg > tg_lims[1] or vbg < bg_lims[0] or vbg > bg_lims[1]:
            pass
        else:
            vtg_new_vec.append(vtg)
            vbg_new_vec.append(vbg)
    return np.array(vtg_new_vec), np.array(vbg_new_vec)

In [ ]:
cst_epsilon0 = cst.epsilon_0
cst_e = cst.e
cst_h = cst.h
cst_phi0 = cst_h/(2*cst_e)
cst_kappa_SiO2 = 3.9
cst_kappa_hBN = 3

def calc_ctg(t_tg):
    """
    Calculates area capacitance for a top gate
    """
    return cst_epsilon0*cst_kappa_hBN/t_tg

def calc_cbg(t_bg, t_bgSiO2):
    """
    Calculates area capacitance for a back gate
    For a metal back gate, set t_bgSiO2 = 0
    """
    return cst_epsilon0*1/(t_bg/cst_kappa_hBN + t_bgSiO2/cst_kappa_SiO2)

def calc_nD(V_tg, V_bg, c_tg, c_bg, n_offset=0, D_offset = 0):
    n = (1/cst_e) * (c_tg*V_tg  + c_bg*V_bg) + n_offset
    D = 0.5*(c_tg*V_tg - c_bg*V_bg)*1/cst_epsilon0 + D_offset
    return n, D


# Device gate-stack constants — fill in per device (meters)
t_tg = <T_TG>          # top-gate hBN thickness
t_bg = <T_BG>          # back-gate hBN thickness
t_bgSiO2 = <T_BGSIO2>  # back-gate SiO2 thickness (0 for metal/graphite back gate)

c_tg = calc_ctg(t_tg)
c_bg = calc_cbg(t_bg, t_bgSiO2)

# Measurement config

In [ ]:
# PATTERN — per-run measurement config: update for the current wiring before
# each measurement block. The contact pairs live here, not in a global table.
meas_config = {
    "sr0": "<drain-current contact note, e.g. 'drain current on 42'>",  # drain
    "sr0_gain": <SR0_GAIN>,  # current preamp gain, e.g. -1e6 (Ithaco)
    # NOTE: record the source contact in "comments", not as a second "sr1" key
    # (duplicate dict keys silently overwrite).
    "sr1": "<contact pair, e.g. '09-01'>",
    "sr2": "<contact pair>",
    "sr3": "<contact pair>",
    "bias_resistor": <BIAS_RESISTOR>,  # e.g. 100e6
    "bias_voltage": sr1.amplitude(),
    "lockin_freq": sr1.frequency(),
    "sr0_sens": sr0.sensitivity(),
    "sr1_sens": sr1.sensitivity(),
    "sr2_sens": sr2.sensitivity(),
    "sr3_sens": sr3.sensitivity(),
    "sample_temperature:": T_sample(),  # file-based (worker logfile); key verbatim (trailing ':') to match this rig's existing metadata
    "Si (V)": k_2410.volt(),
    "comments": "<what/why, drive wiring incl. source contact, Si value, floated contacts>",
}

# Measurement patterns

## PATTERN — 1D gate sweep (~2 min)

In [ ]:
# PATTERN — fill <...> before running. ~2 min for ~100 pts.
s = sweep.AsyncStation(meas_config)

s.fp(sr0.X).fp(sr0.P)
s.fp(sr1.X).fp(sr1.P)
s.fp(sr2.X).fp(sr2.P)
s.fp(sr3.X).fp(sr3.P)

s.sweep(k_bg.volt, np.linspace(<START>, <STOP>, <NPTS>), delay=1)

## PATTERN — density cut along a D-line (~2 min at 101 pts)

In [ ]:
# PATTERN — design a constant-D gate line, clipped to the validators.
# vbg = vtg/<CNP_SLOPE> + <CONST>  — fit the CNP slope from a 0 T gate map first;
# <CONST> sets the displacement field D.
vtg_vec = np.linspace(<VTG_START>, <VTG_STOP>, <NPTS>)
vbg_vec = vtg_vec / <CNP_SLOPE> + <CONST>

vtg_new_vec, vbg_new_vec = sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims)
print("Total number of points", len(vtg_vec))
print("Total number of points (sanitized)", len(vtg_new_vec))

In [ ]:
# PATTERN — density cut along the designed D-line (~2 min at 101 pts).
s = sweep.AsyncStation(meas_config)

s.fp(sr0.X).fp(sr0.P)
s.fp(sr1.X).fp(sr1.P)
s.fp(sr2.X).fp(sr2.P)
s.fp(sr3.X).fp(sr3.P)

s.multisweep([dc205_volt, k_bg.volt], [vtg_new_vec, vbg_new_vec],
             delay=1)

## PATTERN — coarse B-n fan along a D-line (≈2.5 min per 101-pt B-row)

In [ ]:
# PATTERN — gate line for a coarse B-n fan (measure cells 182-184 pattern).
# Budget: ≈1.2 s/pt + ≈30 s per B-row → 33 rows x 101 pts ≈ 1h25m.
vtg_vec = np.linspace(<VTG_START>, <VTG_STOP>, <NPTS>)
vbg_vec = vtg_vec / <CNP_SLOPE> + <CONST>

vtg_new_vec, vbg_new_vec = sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims)
print("Total number of points", len(vtg_vec))
print("Total number of points (sanitized)", len(vtg_new_vec))

In [ ]:
# PATTERN — coarse B-n fan: slow axis B, fast axis the co-swept D-line.
k_2410.volt(<SI_V>)

magnet.mag(<B_START>)

s = sweep.AsyncStation(meas_config)

s.fp(sr0.X).fp(sr0.P)
s.fp(sr1.X).fp(sr1.P)
s.fp(sr2.X).fp(sr2.P)
s.fp(sr3.X).fp(sr3.P)

B_vec = np.linspace(<B_START>, <B_STOP>, <N_B>)
s.multimegasweep([magnet.mag], [B_vec],
                 [dc205_volt, k_bg.volt], [vtg_new_vec, vbg_new_vec],
                 slow_delay=5, fast_delay=1)